In [2]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import os
import sys
from pathlib import Path
import yaml
import zipfile

%load_ext autoreload
%autoreload 2

#project_root = Path.cwd().parent
#sys.path.append(str(project_root))
import functions as fn

#Data raw folder path:
raw_folder = r"C:\Users\ziden\Desktop\Trainings\RNCP-Project\data\raw"
#Data clean folder path:
clean_folder = r"C:\Users\ziden\Desktop\Trainings\RNCP-Project\data\clean"


In [1]:
######################## I. Data Collection #################################


#-----------------------------------------------------------------------------
# 1. FILE1: ARCEP Data QoS source URL: WEB SCRAPING
#-----------------------------------------------------------------------------

url = "https://data.arcep.fr/mobile/mesures_qualite_arcep/2025/Metropole/"

# ----------------------------------------------------------------------------
# 2. Scrape the webpage
# ----------------------------------------------------------------------------

response = requests.get(url)
if (response.status_code) == 200:
    soup = BeautifulSoup(response.text, "html.parser")
else:
    print(f"ERROR! site: {url} can't be scraped!")

#-----------------------------------------------------------------------------
# 3. Find the csv files
#-----------------------------------------------------------------------------

links = soup.find_all("a") #gets all the links of csv files as a list
csv_files = []

for link in links:
    href = link.get("href")

    if href and href.endswith(".csv"):
        csv_files.append({
            "file_name": link.get_text(strip=True),
            "file_url": urljoin(url, href)
        })

files_df = pd.DataFrame(csv_files)
display(files_df)

selected_file = files_df[files_df["file_url"].str.contains("data_habitations.csv", na = False)]
print("Selected file:", selected_file)

file_url = selected_file.iloc[0]['file_url']
print("file url: ", file_url)

# ----------------------------------------------------------------------------
# 4. Download the raw ARCEP 5G mobile QoS file
# ----------------------------------------------------------------------------

raw_folder = r"C:\Users\ziden\Desktop\Trainings\RNCP-Project\data\raw"
file_name = selected_file.iloc[0]["file_name"]
raw_file_path = os.path.join(raw_folder, file_name)

download_response = requests.get(file_url)
with open(raw_file_path, "wb") as f:
    f.write(download_response.content)
print(f"file {file_name} saved successfully.")


,file_name,file_url
0,2025_QoS_Metropole_data_habitations.csv,https://data.arcep.fr/mobile/mesures_qualite_a...
1,2025_QoS_Metropole_data_transports.csv,https://data.arcep.fr/mobile/mesures_qualite_a...
2,2025_QoS_Metropole_voix_habitations.csv,https://data.arcep.fr/mobile/mesures_qualite_a...
3,2025_QoS_Metropole_voix_transports.csv,https://data.arcep.fr/mobile/mesures_qualite_a...


Selected file:                                  file_name  \
0  2025_QoS_Metropole_data_habitations.csv   

                                            file_url  
0  https://data.arcep.fr/mobile/mesures_qualite_a...  
file url:  https://data.arcep.fr/mobile/mesures_qualite_arcep/2025/Metropole/2025_QoS_Metropole_data_habitations.csv
file 2025_QoS_Metropole_data_habitations.csv saved successfully.


In [110]:
#-----------------------------------------------------------------------------
# 1. FILE 2: ARCEP Sites source URL: WEBSCRAPING
#-----------------------------------------------------------------------------

url_sites = "https://data.arcep.fr/mobile/sites/"
response_sites = requests.get(url_sites)

if response_sites.status_code == 200:
    sites_soup = BeautifulSoup(response_sites.text, "html.parser")
else:
    print(f"ERROR! site: {url_sites} can't be scraped!")

#-----------------------------------------------------------------------------
# 2. Find the csv files
#-----------------------------------------------------------------------------

sites_links = sites_soup.find_all("a")

sites_2025_T1 = []
for link in sites_links:
    href = link.get("href")
    text = link.get_text(strip=True)
    
    if text == "2025_T1":
        sites_2025_T1.append({"name": text,
                           "url": urljoin(url_sites,href)
                          })
year_df = pd.DataFrame(sites_2025_T1)

#-----------------------------------------------------------------------------
# 3. Find CSV links
#-----------------------------------------------------------------------------

sites_2025_url = year_df.iloc[0]["url"]
response = requests.get(sites_2025_url)
soup = BeautifulSoup(response.text, "html.parser")

site_csv_files = []
for link in soup.find_all("a"):
    href = link.get("href")
    text = link.get_text(strip=True)

    if href and href.endswith(".csv"):
        site_csv_files.append({
            "file_name": text,
            "file_url" : urljoin(sites_2025_url, href)
        })
site_files_df = pd.DataFrame(site_csv_files)

selected_site = site_files_df[
    site_files_df["file_name"] == "2025_T1_sites_Metropole.csv"
]

site_file_name = selected_site.iloc[0]["file_name"]
site_file_url = selected_site.iloc[0]["file_url"]

# ----------------------------------------------------------------------------
# 5. Download the raw ARCEP sites
# ---------------------------------------------------------------------------- 
site_file_path = os.path.join(raw_folder, site_file_name)
download_response2 = requests.get(site_file_url)
download_response2.raise_for_status() #raise_for_status() so that a failed HTTP request doesn't silently create a bad/empty file.

with open(site_file_path, "wb") as f2:
    f2.write(download_response2.content)
print(f"File {site_file_name} saved successfully.")
site_files_df

File 2025_T1_sites_Metropole.csv saved successfully.


,file_name,file_url
0,2025_T1_sites_5G_historique_comptage.csv,https://data.arcep.fr/mobile/sites/2025_T1/202...
1,2025_T1_sites_Metropole.csv,https://data.arcep.fr/mobile/sites/2025_T1/202...
2,2025_T1_sites_Outremer.csv,https://data.arcep.fr/mobile/sites/2025_T1/202...


In [170]:
#-----------------------------------------------------------------------------
# 1. FILE 3: EXTRACT INSEE POPULATION BY COMMUNE API=Melodi
#-----------------------------------------------------------------------------

url = "https://api.insee.fr/melodi/data/DS_POPULATIONS_REFERENCE"
params = {
    "GEO": "COM",
    "TIME_PERIOD": "2023",
    "POPREF_MEASURE": "PMUN",
    "maxResult": 40000
}

response = requests.get(url, params=params, timeout=60)
#response.status_code

data = response.json()
print(data.keys())

#get the onsee data
observations = data["observations"]
insee_data = [
    {
        "geo": obs["dimensions"]["GEO"],
        "year": obs["dimensions"]["TIME_PERIOD"],
        "measure": obs["dimensions"]["POPREF_MEASURE"],
        "population": obs["measures"]["OBS_VALUE_NIVEAU"]["value"]
    }
    for obs in observations
]

#Convert into data frame
insee_population = pd.DataFrame(insee_data)
#Extract lthe last characters and fill up with zeros if len <5
insee_population["insee_com"] = insee_population["geo"].apply(lambda x: x.split("-")[2].strip()).astype("string").str.zfill(5)

#--------------------------- SAVE TO DATA/RAW/CLEAN ---------------------
#Write the file to data/raw as csv
population_file_name = "insee_com_population.csv"
population_file_path_raw = os.path.join(raw_folder, population_file_name)
population_file_path_clean = os.path.join(clean_folder, population_file_name)
insee_population.to_csv(population_file_path_raw, index=False, encoding = "latin1")
insee_population.to_csv(population_file_path_clean, index=False, encoding = "latin1")

display(insee_population.head())
display(insee_population.dtypes)

dict_keys(['observations', 'identifier', 'paging', 'publisher', 'title'])


,geo,year,measure,population,insee_com
0,2025-COM-01259,2023,PMUN,709.0,01259
1,2025-COM-01216,2023,PMUN,886.0,01216
2,2025-COM-01124,2023,PMUN,703.0,01124
3,2025-COM-01196,2023,PMUN,1261.0,01196
4,2025-COM-02073,2023,PMUN,722.0,02073


geo               str
year              str
measure           str
population    float64
insee_com      string
dtype: object

In [4]:
#-----------------------------------------------------------------------------
# FILE4: cog_geo file: FLAT file extraction: files downloaded to local drive than opened using pd.read_csv

#Open COG file (communes files): Moved to fn.py
# ----------------------------------------------------------------------------
insee_geo = fn.insee_geo_flat_extract()
insee_geo.head()

File 'insee_geo.csv' is successfully saved to 'data/clean' folder.


,insee_com,com_name,insee_dep,dep_name,insee_reg,reg_name
0,01001,ABERGEMENT CLEMENCIAT,01,AIN,84,Auvergne-Rhône-Alpes
1,01002,ABERGEMENT DE VAREY,01,AIN,84,Auvergne-Rhône-Alpes
2,01004,AMBERIEU EN BUGEY,01,AIN,84,Auvergne-Rhône-Alpes
3,01005,AMBERIEUX EN DOMBES,01,AIN,84,Auvergne-Rhône-Alpes
4,01006,AMBLEON,01,AIN,84,Auvergne-Rhône-Alpes
